In [1]:
!pip install -q roboflow ultralytics==8.0.134 segmentation_models_pytorch==0.3.0 onnx onnxruntime onnx-simplifier pytorch-grad-cam albumentations==1.3.1 timm

import os, sys, random, time, math, json, shutil
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
import torchvision.transforms as T
from PIL import Image
from tqdm import tqdm


SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)


  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
ERROR: Ignored the following versions that require a different python version: 8.0.10 Requires-Python >=3.7,<=3.11; 8.0.11 Requires-Python >=3.7,<=3.11; 8.0.12 Requires-Python >=3.7,<=3.11; 8.0.13 Requires-Python >=3.7,<=3.11; 8.0.14 Requires-Python >=3.7,<=3.11; 8.0.15 Requires-Python >=3.7,<=3.11; 8.0.16 Requires-Python >=3.7,<=3.11; 8.0.17 Requires-Python >=3.7,<=3.11; 8.0.18 Requires-Python >=3.7,<=3.11; 8.0.19 Requires-Python >=3.7,<=3.11; 8.0.20 Requires-Python >=3.7,<=3.11; 8.0.21 Requires-Python >=3.7,<=3.11; 8.0.22 Requires-Python >=3.7,<=3.11; 8.0.23 Requires-Python >=3.7,<=3.11; 8.0.24 Requires-Python >=3.7,<=3.11; 8.0.25 Requires-Python >=3.7,<=3.11; 8.0.26 Requires-Python >=3.7,<=3.11; 8.0.27 Requires-Python >=3.7,<=3.11; 8.0.28 Requires-Python >=3.7,<=3.11; 8.0.29 Requires-Python >=3.7,<=3.11; 8.0.30 Requires-Python >=3.7,<=3.11; 8.0.31 Re

In [4]:
!pip install roboflow
from roboflow import Roboflow

ROBOFLOW_API_KEY = "cjWOherJaBOvb1zkqRKJ"
WORKSPACE = "synthetic-data-3ol2y"
PROJECT = "go-positions"
VERSION = 6
EXPORT_FORMAT = "yolov5"

if ROBOFLOW_API_KEY:
    from roboflow import Roboflow
    rf = Roboflow(api_key=ROBOFLOW_API_KEY)
    project = rf.workspace(WORKSPACE).project(PROJECT)
    version = project.version(VERSION)
    out = version.download(EXPORT_FORMAT)
    DATA_DIR = Path(out.location or str(out))
    print("Dataset downloaded to:", DATA_DIR)
else:
    if os.path.exists("/content/Go-Positions-6"):
        DATA_DIR = Path("/content/Go-Positions-6")
        print("Using existing dataset at", DATA_DIR)
    else:
        raise RuntimeError("Нет ROBOFLOW_API_KEY и нет локального датасета. Загрузите датасет в /content/ или задайте ROBOFLOW_API_KEY.")


  Using cached roboflow-1.3.10-py3-none-any.whl.metadata (11 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 250.0/250.0 kB 12.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.8/66.8 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.9/49.9 MB 17.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 77.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 99.6 MB/s eta 0:00:00
  Attempting uninstall: opencv-python-headless
    Found existing installation: opencv-python-headless 4.13.0.92
    Uninstalling opencv-python-headless-4.13.0.92:
      Successfully uninstalled opencv-python-headless-4.13.0.92
  Attempting uninstall: idna
    Found existing installation: idna 3.18
    Uninstalling idna-3.18:
      Successfully uninstalled idna-3.18
loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to Go-Positions-6 in yolov5pytorch:: 100%|██████████| 4812/4812 [00:01<00:00, 4363.90it/s]

Dataset downloaded to: /content/Go-Positions-6


In [5]:
print("DATA_DIR:", DATA_DIR)
for root, dirs, files in os.walk(DATA_DIR):
    print(root)
    print("  dirs:", dirs[:10])
    print("  files:", files[:20])
    break

DATA_DIR: /content/Go-Positions-6
/content/Go-Positions-6
  dirs: ['test', 'valid', 'train']
  files: ['README.roboflow.txt', 'README.dataset.txt', 'data.yaml']


In [6]:
data_yaml = {
    'train': str(DATA_DIR / 'train' / 'images'),
    'val': str(DATA_DIR / 'valid' / 'images'),
    'test': str(DATA_DIR / 'test' / 'images'),
    'names': ['empty','black','white']
}
import yaml
os.makedirs('yolo_data', exist_ok=True)
with open('yolo_data/data.yaml', 'w') as f:
    yaml.dump(data_yaml, f)
print("Wrote yolo_data/data.yaml")


Wrote yolo_data/data.yaml


In [11]:
!git clone https://github.com/ultralytics/yolov5.git
%cd yolov5
!pip install -r requirements.txt
!python train.py --img 640 --batch 16 --epochs 10 --data ../yolo_data/data.yaml --weights yolov5s.pt --name go_yolov5_exp


Cloning into 'yolov5'...
remote: Enumerating objects: 18370, done.
remote: Counting objects: 100% (57/57), done.
remote: Compressing objects: 100% (39/39), done.
remote: Total 18370 (delta 37), reused 18 (delta 18), pack-reused 18313 (from 3)
Receiving objects: 100% (18370/18370), 17.52 MiB | 14.77 MiB/s, done.
Resolving deltas: 100% (12482/12482), done.
/content/yolov5
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.3/41.3 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 36.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 131.1/131.1 kB 16.1 MB/s eta 0:00:00
  Attempting uninstall: urllib3
    Found existing installation: urllib3 2.5.0
    Uninstalling urllib3-2.5.0:
      Successfully uninstalled urllib3-2.5.0
  Attempting uninstall: ultralytics
    Found existing installation: ultralytics 8.0.134
    Uninstalling ultralytics-8.0.134:
      Successfully uninstalled ultralytics-8.0.134


Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
wandb: WARNING ⚠️ wandb is deprecated and will be removed in a future release. See supported integrations at https://github.com/ultralytics/yolov5#integrations.
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results
wandb: Enter your choice: (30 second timeout) 3
wandb: You chose "Don't visualize my results"
wandb: Using W&B in offline mode.
wandb: W&B API key is configured. Use `wandb login --relogin` to force relogin
train: weights=yolov5s.pt, cfg=, data=../yolo_data/data.yaml, hyp=data/hyps/hyp.scratch-low.yaml, epochs=10, batch_size=16, imgsz=640, rect=False, resume=False, nosave=False, noval=False, noautoanchor=False, nopl

In [13]:
import subprocess, shlex, os, glob
weights = 'runs/train/go_yolov5_exp/weights/best.pt'
cmd = f"python export.py --weights {weights} --img 640 --batch 1 --include onnx --simplify --opset 12"
print("Running:", cmd)
ret = subprocess.run(shlex.split(cmd), stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
print(ret.stdout[:2000])

onnx_candidates = glob.glob('**/*.onnx', recursive=True)
onnx_candidates = [p for p in onnx_candidates if 'best' in p or 'export' in p or p.endswith('.onnx')]
print("Found ONNX candidates:", onnx_candidates[:10])
os.makedirs('/content/model_out', exist_ok=True)
if onnx_candidates:
    src = onnx_candidates[-1]
    dst = '/content/model_out/go_yolo.onnx'
    import shutil
    shutil.copy2(src, dst)
    print("Copied", src, "->", dst)
else:
    print("ONNX не найден. Проверьте вывод export.py для ошибок.")


Running: python export.py --weights runs/train/go_yolov5_exp/weights/best.pt --img 640 --batch 1 --include onnx --simplify --opset 12
export: data=data/coco128.yaml, weights=['runs/train/go_yolov5_exp/weights/best.pt'], imgsz=[640], batch_size=1, device=cpu, half=False, inplace=False, keras=False, optimize=False, int8=False, per_tensor=False, dynamic=False, cache=, simplify=True, mlmodel=False, opset=12, verbose=False, workspace=4, nms=False, agnostic_nms=False, topk_per_class=100, topk_all=100, iou_thres=0.45, conf_thres=0.25, include=['onnx']
YOLOv5 🚀 v7.0-505-gf4afe0f5 Python-3.12.13 torch-2.11.0+cu128 CPU

Fusing layers... 
Model summary: 157 layers, 7018216 parameters, 0 gradients, 15.8 GFLOPs

PyTorch: starting from runs/train/go_yolov5_exp/weights/best.pt with output shape (1, 25200, 8) (13.8 MB)
requirements: Ultralytics requirements ['onnx>=1.12.0', 'onnxscript'] not found, attempting AutoUpdate...
Using Python 3.12.13 environment at: /usr
Resolved 10 packages in 404ms
Prepare

In [ ]:
metrics = model.val(data='yolo_data/data.yaml', imgsz=640, batch=16)
print("Validation metrics:", metrics)

os.makedirs('yolo_out', exist_ok=True)
onnx_path = 'yolo_out/go_yolo.onnx'
model.export(format='onnx', imgsz=640, simplify=True, opset=12, file=onnx_path)
print("Exported YOLO ONNX to", onnx_path)

import glob, json
runs = sorted(glob.glob('runs/*/go_yolo_exp*'), key=os.path.getmtime)
if runs:
    run = runs[-1]
    print("Found run:", run)
    csv_path = os.path.join(run, 'metrics.csv')
    if os.path.exists(csv_path):
        import pandas as pd
        df = pd.read_csv(csv_path)
        display(df.head())
